# CpG Site Selection for MethylGPT

This notebook demonstrates the CpG selection pipeline used in MethylGPT.  
Users can modify the trait association threshold to generate custom CpG lists for model training.

## Selection Criteria

CpG sites were selected from the [EWAS Atlas](https://ngdc.cncb.ac.cn/ewas/atlas) based on:
1. **Association with >= 5 traits** at P < 1 x 10^-5 (configurable threshold)
2. **Presence in > 95% of pretraining datasets**
3. **Variance > 0.01** across samples

The EWAS Atlas contains 643,805 associations across **729 unique traits** and **301,524 CpG sites**, spanning diseases (e.g., cancer, diabetes), exposures (e.g., smoking), demographics (e.g., aging, gender), and developmental conditions.

## Rationale for >= 5 Trait Threshold

1. **Sufficient genome coverage**: The selected CpG set provides ~15.8 CpGs/Mb with a median gap of 3.4 kb
2. **Biological relevance**: CpGs associated with multiple traits likely have broader connections to pathological processes, making them appropriate for downstream disease-related tasks
3. **Computational feasibility**: Balances model input size with available GPU memory

## Data Source

Download the EWAS Atlas associations file from:  
https://ngdc.cncb.ac.cn/ewas/atlas  
(Navigate to **Downloads** -> **Association** -> TSV file)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Step 1: Load EWAS Atlas Associations

The EWAS Atlas TSV file contains columns: `probe_ID`, `trait`, `p_value`, `study_ID`, etc.  
We count the number of distinct traits associated with each CpG site.

In [ ]:
# --- USER CONFIG ---
EWAS_ATLAS_PATH = "EWAS_Atlas_associations.tsv"  # Path to downloaded EWAS Atlas file
OUTPUT_DIR = "./"  # Where to save the output probe_ids CSV
# -------------------

# Load EWAS Atlas (only columns we need)
ewas_df = pd.read_csv(EWAS_ATLAS_PATH, sep="\t", usecols=["probe_ID", "trait"])
print(f"Total associations: {len(ewas_df):,}")
print(f"Unique CpG sites:  {ewas_df['probe_ID'].nunique():,}")
print(f"Unique traits:     {ewas_df['trait'].nunique():,}")

In [ ]:
# Count number of unique traits per CpG
trait_counts = ewas_df.groupby("probe_ID")["trait"].nunique()
trait_counts.name = "n_traits"

print(f"\nTrait count distribution across {len(trait_counts):,} CpGs:")
print(trait_counts.describe())

## Step 2: Compare Thresholds

We evaluate how different trait-count thresholds (1, 3, 5, 10) affect the number of selected CpGs, genome coverage, and estimated memory requirements.

In [ ]:
THRESHOLDS = [1, 3, 5, 10]
GENOME_SIZE_MB = 3088  # Human genome size in Mb (hg38)

# Medium architecture: ~300 bytes per CpG per sample for a forward pass (empirical estimate)
BYTES_PER_CPG_PER_SAMPLE = 300

results = []
for t in THRESHOLDS:
    n_cpgs = (trait_counts >= t).sum()
    density = n_cpgs / GENOME_SIZE_MB
    gb_per_sample = n_cpgs * BYTES_PER_CPG_PER_SAMPLE / 1e9
    results.append({
        "Threshold": f">= {t} traits",
        "CpG Count": n_cpgs,
        "% of EWAS CpGs": f"{n_cpgs / len(trait_counts) * 100:.1f}%",
        "CpGs/Mb": f"{density:.1f}",
        "Est. GB/sample (medium)": f"{gb_per_sample:.2f}",
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

## Step 3: Visualize Threshold Effects

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# --- Left: Trait count distribution ---
ax = axes[0]
ax.hist(trait_counts.values, bins=np.arange(0.5, 51.5, 1), color="#4575b4", edgecolor="black", alpha=0.8)
ax.axvline(x=5, color="#d73027", linestyle="--", linewidth=2, label="Threshold = 5")
ax.set_xlabel("Number of EWAS Trait Associations")
ax.set_ylabel("Number of CpG Sites")
ax.set_title("Distribution of Trait Associations per CpG")
ax.set_xlim(0, 50)
ax.legend()

# --- Right: CpG count vs threshold ---
ax = axes[1]
x_range = np.arange(1, 21)
cpg_counts = [(trait_counts >= t).sum() for t in x_range]

ax.plot(x_range, cpg_counts, "o-", color="#4575b4", linewidth=2, markersize=6,
        markerfacecolor="white", markeredgecolor="#4575b4", markeredgewidth=1.5)
ax.fill_between(x_range, cpg_counts, alpha=0.15, color="#4575b4")

# Mark the default threshold
idx5 = np.where(x_range == 5)[0][0]
ax.scatter([5], [cpg_counts[idx5]], color="#d73027", s=120, zorder=5,
           label=f"Default (>= 5): {cpg_counts[idx5]:,} CpGs")
ax.axvline(x=5, color="#d73027", linestyle="--", linewidth=1.5, alpha=0.5)

ax.set_xlabel("Minimum Trait Associations")
ax.set_ylabel("Number of CpG Sites Selected")
ax.set_title("CpG Selection vs Threshold")
ax.legend()

plt.tight_layout()
plt.savefig("cpg_threshold_analysis.pdf", bbox_inches="tight")
plt.savefig("cpg_threshold_analysis.png", dpi=600, bbox_inches="tight")
plt.show()

## Step 4: Generate Custom CpG List

Select your desired threshold and export a `probe_ids.csv` file compatible with MethylGPT's preprocessing pipeline.

In [ ]:
# --- USER CONFIG: Change this threshold as needed ---
MIN_TRAITS = 5  # Default MethylGPT threshold
# ---------------------------------------------------

selected_cpgs = trait_counts[trait_counts >= MIN_TRAITS].index.tolist()
selected_cpgs.sort()

print(f"Selected {len(selected_cpgs):,} CpGs with >= {MIN_TRAITS} trait associations")

# Save in the format expected by MethylGPT
output_path = f"{OUTPUT_DIR}/probe_ids_ewas_min{MIN_TRAITS}.csv"
out_df = pd.DataFrame({"illumina_probe_id": selected_cpgs})
out_df.to_csv(output_path, index=True)

print(f"Saved to: {output_path}")
print(f"\nFirst 10 CpGs:")
print(out_df.head(10).to_string())

## Step 5 (Optional): Intersect with Dataset Availability

The final MethylGPT CpG list (49,156 sites) applies two additional filters:
- **Presence in > 95% of pretraining datasets** (to minimize missing values)
- **Variance > 0.01** across samples (to exclude invariant sites)

If you have your own methylation dataset, you can apply these filters as shown below.

In [ ]:
# Example: Apply availability and variance filters to your own data
#
# # Load your methylation matrix (samples x CpGs)
# beta_matrix = pd.read_csv("your_beta_matrix.csv", index_col=0)
#
# # Filter 1: Keep CpGs present in > 95% of samples
# availability = beta_matrix.notna().mean(axis=0)
# available_cpgs = availability[availability > 0.95].index
#
# # Filter 2: Keep CpGs with variance > 0.01
# variance = beta_matrix[available_cpgs].var(axis=0)
# variable_cpgs = variance[variance > 0.01].index
#
# # Intersect with EWAS-selected CpGs
# final_cpgs = sorted(set(selected_cpgs) & set(variable_cpgs))
# print(f"Final CpG set: {len(final_cpgs)} sites")
#
# # Save
# pd.DataFrame({"illumina_probe_id": final_cpgs}).to_csv("probe_ids_custom.csv", index=True)

## Summary

| Step | Filter | Purpose |
|------|--------|---------|
| 1 | EWAS trait count >= 5 | Select biologically relevant CpGs associated with multiple phenotypes |
| 2 | Present in > 95% of datasets | Ensure data availability across training samples |
| 3 | Variance > 0.01 | Exclude invariant sites with no discriminative signal |

The output `probe_ids.csv` can be used directly with the MethylGPT preprocessing and training pipelines.  
See `tutorials/pretraining/` for the full pretraining workflow.